# 🤖 Synthèse de l'Intégration du LLM HuggingFace (Mistral-7B)

> Ce notebook documente en détail les étapes d'intégration de l'assistant d'explication de commandes IA via l'API Inference de HuggingFace, ainsi que les modifications apportées aux différents fichiers du projet.

---

## 🚀 1. Étapes de l'Intégration

L'intégration s'est déroulée en 4 phases :
1. **Configuration de l'environnement** : Création/mise à jour du fichier `.env` pour supporter les variables d'API HuggingFace et de cache.
2. **Écriture du service d'explication** : Implémentation complète de `src/services/explanation.py` avec cache mémoire MD5, appels réseau HTTP vers Mistral-7B, et règles de fallback en cas d'erreur ou d'absence de clé.
3. **Raccordement au pipeline** : Modification de `src/services/recommendation.py` pour substituer les explications textuelles statiques par des appels au nouveau service dynamique.
4. **Validation et tests de robustesse** : Test de l'API FastAPI avec serveur local actif, prouvant le bon fonctionnement de la couche de repli (fallback) lorsque le jeton API est vide.


## 📂 2. Modifications Apportées par Fichier

### 📁 A. `.env` (Variables de Configuration)
- Ajout de la clé `HF_TOKEN` pour stocker le jeton utilisateur.
- Définition du modèle par défaut (`mistralai/Mistral-7B-Instruct-v0.3`).
- Configuration du temps de vie du cache (`CACHE_TTL=86400` secondes, soit 24h).

### 📁 B. `src/services/explanation.py` (Nouveau Service)
- **`_get_cache_key(...)`** : Génère une clé unique à l'aide d'un hash MD5 combinant le client, le produit et le mois en cours.
- **`_call_huggingface_api(...)`** : Réalise l'appel POST HTTP vers HuggingFace Inference API avec timeout, gestion des erreurs et parsing de la réponse.
- **`_rule_based_explanation(...)`** : Génère des explications intelligentes en français à partir de règles métiers sur la tendance historique, la récence et la fidélité si le LLM n'est pas actif.
- **`explain_suggestion(...)`** : Fonction principale orchestrant la recherche en cache, l'appel LLM avec prompt en français structuré, et le repli sur les règles.

### 📁 C. `src/services/recommendation.py` (Pipeline Connecté)
- Remplacement des blocs de construction de chaînes d'explication statiques par un import et un appel dynamique à la fonction `explain_suggestion` :
```python
from src.services.explanation import explain_suggestion
explication = explain_suggestion(
    client_id=client_id,
    code_article=str(row["code_article"]),
    designation=designation,
    categorie=cat,
    quantite_suggeree=sugg_qty,
    score_confiance=prob,
    recency_days=int(recency),
    frequency=freq,
    trend=float(row.get("trend", 0)),
    is_new_product=is_new,
)
```


## 🧪 3. Démonstration & Test d'API en Direct

Exécutez la cellule ci-dessous pour tester l'API de recommandation locale et afficher les explications générées en temps réel.

In [1]:
import requests
import json

url = 'http://127.0.0.1:8000/api/recommend'
payload = {
    'client_id': 'CLT091206',
    'commercial_id': 'COMMERCIAL_LSAT',
    'config': {
        'nb_suggestions': 2
    }
}

try:
    response = requests.post(url, json=payload, timeout=5)
    print(f"Code statut HTTP : {response.status_code}\n")
    data = response.json()
    for i, sug in enumerate(data.get('suggestions', []), 1):
        print(f"Suggestion #{i} : {sug['designation']}")
        print(f"  -> Quantité suggérée : {sug['quantite_suggeree']} unités")
        print(f"  -> Explication générée : {sug['explication']}\n")
except Exception as e:
    print(f"Le serveur FastAPI local est-il démarré ? Erreur de connexion : {e}")

Code statut HTTP : 200

Suggestion #1 : REDMI NOTE 13 MIDNIGHT BLACK 6/128GB
  -> Quantité suggérée : 16 unités
  -> Explication générée : Tendance à la hausse détectée pour cet article. Volume de 16 unités suggéré pour répondre à la demande (Confiance 99%).

Suggestion #2 : REDMI 14C MIDNIGHT BLACK 4/128GB
  -> Quantité suggérée : 7 unités
  -> Explication générée : Baisse légère de la demande historique. Quantité prudente recommandée de 7 unités (Confiance 99%).

